<a href="https://colab.research.google.com/github/edwardoughton/spatial_computing/blob/main/11_01_Shortest_Path_Routing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 10: Shortest-Path Routing

Welcome to this class on shortest-path routing. Today we build on our work with lists, dictionaries, loops, and GeoPandas to represent a network and find routes through it.

**Our question:** How can we find the shortest route between two places, and how does this differ from connecting every place with the least total infrastructure?

By the end of class, you should be able to:

- Describe nodes, edges, paths, and edge weights.
- Build and inspect a small network using NetworkX.
- Find and map a shortest path using Dijkstra's algorithm.
- Fit a minimum spanning tree using Kruskal's algorithm and explain its purpose.
- Export spatial results and discuss the limits of our model.

Run the examples in order. Each code cell introduces a small step. Exercise cells are for your own attempts. The networks below are fictional teaching examples, so no external data download is needed.

## A Basic Intro to Graph Theory

A **graph** describes connections between objects. A transport network is one example: junctions are **nodes** (also called vertices), and road sections are **edges**.

A **path** is a sequence of nodes joined by edges. A **weight** records the cost of an edge, such as distance in kilometers or travel time in minutes. The cost of a path is the sum of its edge weights.

- An **undirected** graph allows travel both ways along an edge.
- A **directed** graph can represent one-way travel.
- A **connected** undirected graph has a path between every pair of nodes.

Our first example is undirected. All weights represent distance in kilometers.

## Set up our tools

NetworkX works with networks. Matplotlib draws our figures. GeoPandas and Shapely will help us create spatial features later. Run this installation cell once in a fresh Colab session.

In [ ]:
%pip -q install networkx matplotlib geopandas

Import the packages. As in previous weeks, `as` gives a package a shorter name.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import LineString

## Define our network

We will connect five fictional places, labeled A to E. Start with an empty, undirected `Graph`. Calling `.add_nodes_from()` adds the labels in our list.

In [ ]:
G = nx.Graph()
G.add_nodes_from(["A", "B", "C", "D", "E"])

Each tuple below contains `(start node, end node, distance)`. A tuple keeps these three related values together. These distances are invented, rather than calculated from the drawing.

In [ ]:
edges = [
    ("A", "B", 2),
    ("A", "C", 5),
    ("B", "C", 2),
    ("B", "D", 6),
    ("C", "D", 1),
    ("C", "E", 7),
    ("D", "E", 3),
]

A `for` loop unpacks each tuple into three variables. `.add_edge()` adds a connection, and `distance_km` names the attribute holding its weight. We will use this exact attribute name when finding routes.

In [ ]:
for start, end, distance in edges:
    G.add_edge(start, end, distance_km=distance)

Inspect the number of nodes and edges, then check whether the graph is connected.

In [ ]:
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Connected:", nx.is_connected(G))

## Draw the network

This dictionary places labels on our diagram. The coordinate pairs are drawing positions, with no geographic CRS. They do not determine the edge weights.

In [ ]:
positions = {
    "A": (0, 1),
    "B": (1, 2),
    "C": (2, 1),
    "D": (3, 2),
    "E": (4, 1),
}

Get the distance labels from the graph. Then draw the nodes, connections, and distances. Each edge label is in kilometers.

In [ ]:
distance_labels = nx.get_edge_attributes(G, "distance_km")
fig, ax = plt.subplots(figsize=(9, 5))
nx.draw(G, positions, ax=ax, with_labels=True, node_color="lightblue")
nx.draw_networkx_edge_labels(G, positions, ax=ax, edge_labels=distance_labels)
ax.set_title("Our fictional network: edge distances in km")
plt.show()

## Exercise: predict a route

Before running a routing algorithm:

1. Write down two possible paths from A to E.
2. Add the distances along each path.
3. Predict the shortest path and its total distance.

Remember: the fewest edges may not give the shortest distance.

In [ ]:
# Enter your attempt here


## Dijkstra's Shortest-Path Algorithm

Dijkstra's algorithm finds a path with the smallest total weight. We use nonnegative weights; negative distances would not make sense here.

Starting at A, think of the algorithm as keeping a best distance found so far:

1. Give A a distance of zero. Other nodes begin with an unknown distance (infinity).
2. Select the unvisited node with the smallest known distance.
3. Try reaching its neighbors through this node. Keep any improvement.
4. Mark the selected node as visited and repeat until the destination is selected, or no reachable nodes remain.

For our example, the best distances become A = 0, B = 2, C = 4, D = 5, and E = 8. Reaching C through B improves on the direct A-C edge of 5 km.

We will use NetworkX's implementation so we can focus on the network and the result.

Specify our origin and destination. `.dijkstra_path()` returns the ordered list of nodes. `weight="distance_km"` tells it which edge attribute to minimize.

In [ ]:
origin = "A"
destination = "E"
route = nx.dijkstra_path(G, origin, destination, weight="distance_km")
print("Route:", route)

Find the total distance using the same weight attribute. The expected result is 8 km: A-B-C-D-E.

In [ ]:
route_km = nx.dijkstra_path_length(G, origin, destination, weight="distance_km")
print("Route distance (km):", route_km)

Let us check the result one edge at a time. `route[:-1]` leaves out the last node; `route[1:]` leaves out the first. `zip()` pairs these lists into consecutive connections.

In [ ]:
route_edges = list(zip(route[:-1], route[1:]))
print(route_edges)

`G.edges[start, end]` gives us the attributes of one edge. Use the distance attribute to print each part of the journey.

In [ ]:
for start, end in route_edges:
    distance = G.edges[start, end]["distance_km"]
    print(start, "to", end, ":", distance, "km")

Draw the whole network in gray and our route in red. Using the same positions makes the figures easy to compare.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
nx.draw(G, positions, ax=ax, with_labels=True, edge_color="lightgray")
nx.draw_networkx_edges(G, positions, ax=ax, edgelist=route_edges, edge_color="red", width=3)
nx.draw_networkx_edge_labels(G, positions, ax=ax, edge_labels=distance_labels)
ax.set_title("Shortest route from A to E (red): 8 km")
plt.show()

## Exercise: change the destination

Find the shortest route from A to D and print its total distance. Use new variable names so that `route` still contains our A-to-E journey.

Which is shorter: taking A-B-D or traveling through C? Explain your answer using the edge weights.

In [ ]:
# Enter your attempt here


## What if a road closes?

Create a copy with `.copy()` so our original graph remains available. Remove the connection between C and D, then find a new route.

In [ ]:
closed_graph = G.copy()
closed_graph.remove_edge("C", "D")
closed_route = nx.dijkstra_path(closed_graph, origin, destination, weight="distance_km")
print("Route after closure:", closed_route)

In [ ]:
closed_km = nx.dijkstra_path_length(closed_graph, origin, destination, weight="distance_km")
print("New distance (km):", closed_km)
print("Extra distance (km):", closed_km - route_km)

The new route is 11 km, adding 3 km. Now suppose both connections to E close. `.has_path()` lets us check whether travel is possible before requesting a route.

In [ ]:
isolated_graph = closed_graph.copy()
isolated_graph.remove_edge("C", "E")
isolated_graph.remove_edge("D", "E")
print("Can we reach E?", nx.has_path(isolated_graph, origin, destination))

## Exercise: interpret the model

Why would a real navigation app need more information than this graph? Discuss one-way roads, speed, traffic, and access restrictions.

If weights represented minutes, what would the algorithm minimize? Would that necessarily be the shortest distance?

In [ ]:
# Enter your explanation here as comments


## Connecting every node: a different question

A **minimum spanning tree (MST)** connects every node in a connected, undirected graph while minimizing the sum of the selected edge weights. It contains no cycles (closed loops).

A shortest path minimizes travel cost between an origin and a destination. An MST minimizes the total weight needed to connect all nodes. The route between two nodes within an MST may be longer than their shortest path in the full graph.

For infrastructure planning, an MST can represent a simplified minimum-length layout. It does not provide backup connections if a link fails.

## Kruskal's Minimum Spanning Tree Algorithm

Kruskal's algorithm:

1. Sort the edges from smallest to largest weight.
2. Add the next edge only if it does not create a cycle.
3. Stop when all nodes are connected.

A connected graph with **V** nodes has **V - 1** edges in its spanning tree. Our five-node graph should therefore produce four tree edges.

NetworkX handles the cycle checks for us. If the input graph is disconnected, the result is a spanning forest rather than one connected tree.

In [ ]:
tree = nx.minimum_spanning_tree(G, weight="distance_km", algorithm="kruskal")
print("Tree edges:", list(tree.edges()))
print("Number of tree edges:", tree.number_of_edges())

Add the distance attributes of the selected edges. `size(weight=...)` gives the sum of edge weights; without a weight it would count edges.

In [ ]:
tree_km = tree.size(weight="distance_km")
print("Total network length (km):", G.size(weight="distance_km"))
print("MST length (km):", tree_km)

Plot the selected tree links in blue over the full graph. The total tree length is 8 km in this example.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
nx.draw(G, positions, ax=ax, with_labels=True, edge_color="lightgray")
nx.draw_networkx_edges(tree, positions, ax=ax, edge_color="blue", width=3)
nx.draw_networkx_edge_labels(G, positions, ax=ax, edge_labels=distance_labels)
ax.set_title("Minimum spanning tree (blue)")
plt.show()

## Exercise: route versus tree

Find the shortest route from A to C in both `G` and `tree`. In this example the answers match.

Now explain why this one comparison does not prove that an MST always preserves shortest routes. We will test a spatial example next.

In [ ]:
# Enter your attempt here


## A spatial example near campus

Let us create four fictional sites near Fairfax, Virginia. These are illustrative coordinates, not surveyed facilities or actual roads.

A dictionary stores each column. `points_from_xy()` builds points from longitude (x) and latitude (y). We assign EPSG:4326 because these input coordinates are in degrees.

In [ ]:
site_data = {
    "site_id": ["P", "Q", "R", "S"],
    "longitude": [-77.310, -77.300, -77.300, -77.310],
    "latitude": [38.830, 38.830, 38.838, 38.838],
}

In [ ]:
site_points = gpd.points_from_xy(site_data["longitude"], site_data["latitude"])
sites = gpd.GeoDataFrame(site_data, geometry=site_points, crs="EPSG:4326")
sites.head()

Project to UTM Zone 18N (EPSG:32618), which uses meters and is suitable for our small Fairfax example. Do not calculate lengths directly from longitude and latitude degrees.

In [ ]:
sites = sites.to_crs("EPSG:32618")
print("Projected CRS:", sites.crs)

Use `site_id` as the index so we can look up each point by its label. Our candidate connections form a four-sided network. Each link will be a straight line, not a road route.

In [ ]:
sites = sites.set_index("site_id")
connections = [("P", "Q"), ("Q", "R"), ("R", "S"), ("S", "P")]

Create an empty graph and add site IDs as nodes. The node IDs must match the IDs used by our connections.

In [ ]:
spatial_graph = nx.Graph()
spatial_graph.add_nodes_from(sites.index)

For each connection, look up its two points and join them with a `LineString`. Its `.length` is in meters because the points are projected. Store both the distance and geometry on the edge.

In [ ]:
for start, end in connections:
    start_point = sites.loc[start, "geometry"]
    end_point = sites.loc[end, "geometry"]
    line = LineString([start_point, end_point])
    spatial_graph.add_edge(start, end, distance_m=line.length, geometry=line)

Find a route from P to S and its distance. This time the weight name is `distance_m`. Always match the name stored on the edges.

In [ ]:
spatial_route = nx.dijkstra_path(spatial_graph, "P", "S", weight="distance_m")
spatial_distance = nx.dijkstra_path_length(spatial_graph, "P", "S", weight="distance_m")
print("Route:", spatial_route)
print("Distance (km):", round(spatial_distance / 1000, 3))

Pair consecutive route nodes again. Then collect each selected edge geometry into a list.

In [ ]:
spatial_route_edges = list(zip(spatial_route[:-1], spatial_route[1:]))
route_lines = []
for start, end in spatial_route_edges:
    route_lines.append(spatial_graph.edges[start, end]["geometry"])

A GeoDataFrame gives each route segment its own row. Keep the projected CRS so we can calculate a distance column.

In [ ]:
route_layer = gpd.GeoDataFrame(geometry=route_lines, crs=sites.crs)
route_layer["distance_m"] = route_layer.geometry.length
route_layer.head()

Collect every candidate link as a GeoSeries for our map background.

In [ ]:
all_lines = []
for start, end in connections:
    all_lines.append(spatial_graph.edges[start, end]["geometry"])
network_layer = gpd.GeoSeries(all_lines, crs=sites.crs)

Draw the network, route, and sites on one set of axes. The legend identifies the layers; axis values are UTM coordinates in meters. No internet basemap is required.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
network_layer.plot(ax=ax, color="gray", linewidth=2, label="Candidate links")
route_layer.plot(ax=ax, color="red", linewidth=4, label="Shortest route")
sites.plot(ax=ax, color="black", markersize=45, label="Fictional sites")

Add labels and finish the same figure. Run this cell immediately after the previous one; `fig` and `ax` refer to that map.

In [ ]:
for site_id, row in sites.iterrows():
    ax.annotate(site_id, (row.geometry.x, row.geometry.y), xytext=(6, 6), textcoords="offset points")
ax.set_title("Fictional Fairfax links: shortest route from P to S")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.legend()
fig.tight_layout()
display(fig)
plt.close(fig)

## Fit a spatial minimum spanning tree

Use the same candidate links and meter weights. The MST connects all four sites with three edges. It minimizes length among these candidate links; it cannot choose a connection we did not include.

In [ ]:
spatial_tree = nx.minimum_spanning_tree(spatial_graph, weight="distance_m", algorithm="kruskal")
print("Tree edges:", list(spatial_tree.edges()))
print("Tree length (km):", round(spatial_tree.size(weight="distance_m") / 1000, 3))

Compare travel from P to S in the full network and in the tree. Here the tree removes the direct P-S connection, so traveling within it requires a detour.

In [ ]:
tree_route = nx.dijkstra_path(spatial_tree, "P", "S", weight="distance_m")
tree_route_m = nx.dijkstra_path_length(spatial_tree, "P", "S", weight="distance_m")
print("Route within tree:", tree_route)
print("Full-network route (m):", round(spatial_distance))
print("Route within tree (m):", round(tree_route_m))

Collect the tree geometries and create another GeoDataFrame for mapping or export.

In [ ]:
tree_lines = []
for start, end in spatial_tree.edges():
    tree_lines.append(spatial_tree.edges[start, end]["geometry"])
tree_layer = gpd.GeoDataFrame(geometry=tree_lines, crs=sites.crs)
tree_layer["distance_m"] = tree_layer.geometry.length

## Export our results

Save each output to a separate GeoPackage so it can be opened in QGIS. `reset_index()` returns the site IDs to an ordinary column. These commands write to the current notebook folder and replace these named outputs when rerun.

In Colab, use the Files panel to download the outputs before your session ends.

In [ ]:
sites.reset_index().to_file("week10_sites.gpkg", driver="GPKG")
route_layer.to_file("week10_shortest_route.gpkg", driver="GPKG")
tree_layer.to_file("week10_minimum_spanning_tree.gpkg", driver="GPKG")

## Task 1: map the tree

Plot the candidate links in gray, tree links in blue, and sites in black. Use `tree_layer`, `network_layer`, and `sites`. Include site labels, a title, and a legend.

Explain why removing a single tree edge disconnects part of the network. How could a planner provide backup connections?

In [ ]:
# Enter your attempt here


## Task 2: change the candidate network

Make a copy of `spatial_graph`. Add a straight connection from P to R, using the point lookup and `LineString` pattern above. Store both `geometry` and `distance_m`.

Find the P-to-R shortest path before and after adding the edge. Print both distances in kilometers. Does the new link reduce travel distance? Why might a real project be unable to build it?

In [ ]:
# Enter your attempt here


## Task 3: your own sites

Create four to six fictional sites in a small area of your choosing. If you use real locations, record the source. Choose a projected CRS appropriate for that area with meter units.

- Define a connected set of candidate links and calculate their lengths.
- Find a shortest route between two sites and print its distance in kilometers.
- Fit an MST and print the total length of its selected edges.
- Map the route and tree, then export the results to GeoPackage.
- Explain two assumptions that limit what a planner can conclude.

Start with our four-site example and change one step at a time. Straight lines ignore terrain, buildings, rivers, rights of way, and road access. Crossing lines are not connected unless the graph includes a node and split edges at the crossing.

In [ ]:
# Enter your attempt here


## Check your understanding

1. What do nodes, edges, and weights represent in each example?
2. Why must we specify `weight` when finding a weighted shortest path?
3. How does a shortest path differ from a minimum spanning tree?
4. Why did we project our coordinates before calculating lengths?
5. What changes would be needed to represent one-way roads or travel time?

## Further reading

- [NetworkX: Dijkstra's shortest path](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.shortest_paths.weighted.dijkstra_path.html)
- [NetworkX: minimum spanning tree](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.tree.mst.minimum_spanning_tree.html)

These are the library implementations used in our examples. We focus on interpreting their inputs and outputs rather than writing the algorithms from scratch.